<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 40 Ray tracer

The notebook judges the program of the ray tracer that the shader of the oblique view evaluates: the
postfix order of the tree, the binary operations, the inverse matrices, the Lipschitz factors, the
materials and the bound on the number of nodes.

In [ ]:
max_nodes = 512

In [ ]:
import sys, json, math
sys.path.insert(0, '..')
from harness import *

In [ ]:
build_devtool()

## 1. The postfix order

Every body pushes its distance, every operation combines the two topmost entries. The top level is a
union of the items that belong to no group, and an empty group is a node of code 0.

In [ ]:
p = devtool('program', ','.join(map(str, devtool('example', 'werkstueck')['packed'])))
check(p['codes'] == [5, 3, 102], 'the work piece is cuboid, cylinder, difference', p['codes'])
p = devtool('program', packed(item('SPHERE'), item('CUBE'), item('CONE')))
check(p['codes'] == [1, 4, 101, 2, 101], 'three items at the top are united two at a time', p['codes'])
p = devtool('program', packed(item('UNION'), item('SPHERE')))
check(p['codes'] == [0, 1, 101], 'an empty group is the node of nothing', p['codes'])
p = devtool('program', packed())
check(p['codes'] == [0], 'an empty scene is the node of nothing', p['codes'])

## 2. The stack stays small

A group of many children takes as many entries of the stack as it nests deep, so the fixed stack of
the shader runs any scene.

In [ ]:
def depth(codes):
    top = most = 0
    for c in codes:
        top = top - 1 if c >= 101 else top + 1
        most = max(most, top)
    return most
flat = devtool('program', packed(*[item('SPHERE', (i, 0, 0)) for i in range(40)]))
check(depth(flat['codes']) == 2, 'forty spheres at the top need two entries', depth(flat['codes']))

## 3. The inverse matrices and the Lipschitz factors

A node holds the inverse of the matrix of its body, so the product of both is the identity, and the
smallest scale of the body, which keeps the distance of a scaled body a safe step.

In [ ]:
one = packed(item('CUBOID', (1, 2, 3), (2, 5, 0.5), (10, 20, 30)))
node = devtool('program', one)['nodes'][0]
m = devtool('matrix', one, 0)
inv = node[12:28]
A = [[m[c * 4 + r] for c in range(4)] for r in range(4)]
B = [[inv[c * 4 + r] for c in range(4)] for r in range(4)]
I = [[sum(A[i][k] * B[k][j] for k in range(4)) for j in range(4)] for i in range(4)]
check(close(I, [[1 if i == j else 0 for j in range(4)] for i in range(4)], 1e-3), 'matrix times inverse is the identity', I)
check(close(node[3], 0.5), 'the Lipschitz factor is the smallest scale', node[3])

## 4. The materials

A body wears its own texture, else the texture of the nearest textured group, else its colour.

In [ ]:
textures = devtool('textures')
gold = next(t for t in textures if t['name'] == 'Gold_Metal')
glass = next(t for t in textures if t['name'] == 'Ruby_Glass')
scene = packed(item('UNION', texture=glass['index']), item('SPHERE', parent=0), item('CUBE', parent=0, texture=gold['index']), item('CONE', color=(0.1, 0.2, 0.3)))
nodes = [n for n in devtool('program', scene)['nodes'] if n[0] < 101]
check(close(nodes[0][4:7], glass['base']) and nodes[0][11] == 2, 'the sphere wears the glass of its group')
check(close(nodes[1][4:7], gold['base']) and nodes[1][11] == 1, 'the cube wears its own gold')
check(close(nodes[2][4:7], [0.1, 0.2, 0.3]) and nodes[2][11] == 0, 'the cone wears its colour')
check(len(textures) == 43, 'the table holds the 43 textures of version 1.3')

## 5. Instances

An instance becomes a union of copies of the children of its declaration.

In [ ]:
p = devtool('program', packed(item('DECLARE'), item('CUBE', parent=0), item('INSTANCE', position=(4, 0, 0), source=0)))
check(p['codes'] == [4, 4, 101], 'the declaration and its instance both draw their cube', p['codes'])
moved = p['nodes'][1]
check(close(moved[12 + 12], -4.0), 'the copy of the instance is moved by 4 along x', moved[24])

## 6. The bound on the nodes

In [ ]:
big = devtool('program', packed(*[item('SPHERE', (i % 30, i // 30, 0)) for i in range(400)]))
check(big['count'] == max_nodes, f'a scene of 400 spheres is cut to {max_nodes} nodes', big['count'])